In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
import dlt
from pyspark.sql.functions import col, when, lit, regexp_extract, current_timestamp, to_date, sum, count
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType


@dlt.table(
    comment="Bronze delta table for staging"
)
def bronze_orders():
    source_path = "abfss://raw@sdattastore12.dfs.core.windows.net/sales/"
    schema_location = "abfss://raw@sdattastore12.dfs.core.windows.net/sdatta/"

    raw_stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("pathGlobFilter", "*.csv")
        .load(source_path)
    )

    bronze_df = (
        raw_stream
        .withColumn("sale_id", col("sale_id").cast("int"))
        .withColumn("product_id", col("product_id").cast("int"))
        .withColumn("quantity", col("quantity").cast("int"))
        .withColumn("sales_amount", col("sales_amount").cast("double"))
        .withColumn("sale_date", col("sale_date").cast("timestamp"))
    )

    return bronze_df


@dlt.table(
    comment="Products delta table for staging"
)
def products_table():
    product_df = spark.read.option("multiline", "true").json(
        "abfss://raw@sdattastore12.dfs.core.windows.net/products/products.json"
    )
    return product_df


@dlt.table(
    comment="Silver delta table for transformations"
)
@dlt.expect("valid_quantity", "quantity > 0")
@dlt.expect("valid_sales_amount", "sales_amount > 0")
def silver_orders():
    bronze_df = dlt.read("bronze_orders")
    product_df = dlt.read("products_table")

    silver_df = bronze_df.join(product_df, "product_id", "left")

    silver_df = silver_df.withColumn(
        "has_discount", when(col("discount_rate") > 0, lit(True)).otherwise(lit(False))
    )

    silver_df = silver_df.withColumn(
        "unit_price", when(col("quantity") > 0, col("sales_amount") / col("quantity")).otherwise(lit(0))
    )

    silver_df = silver_df.withColumn(
        "revenew_tier",
        when(col("sales_amount") > 1000, "High")
        .when(col("sales_amount") > 500, "Medium")
        .otherwise("Low"),
    )

    silver_df = silver_df.withColumn(
        "quantity_load",
        when(col("quantity") > 10, "Bulk")
        .when(col("quantity") > 5, "Medium")
        .otherwise("Low"),
    )

    silver_df = silver_df.withColumn(
        "store_number", regexp_extract(col("store_id"), r"Store_(\d+)", 1).cast("int")
    )

    silver_df = silver_df.fillna(
        {
            "discount_rate": 0,
            "unit_price": 0,
        }
    )

    silver_df = silver_df.withColumn("_ingest_ts", current_timestamp())

    silver_df = silver_df.dropDuplicates(["sale_id"])

    return silver_df


@dlt.table(
    comment="Gold delta table for aggregations by region and date"
)
def daily_region_sales():
    silver_df = dlt.read("silver_orders")

    silver_df = silver_df.withColumn("sales_date_only", to_date(col("sale_date")))

    gold_agg1 = silver_df.groupBy("region", "sales_date_only").agg(
        sum("sales_amount").alias("total_sales"),
        count("sale_id").alias("total_orders"),
    )

    return gold_agg1


@dlt.table(
    comment="Gold delta table for category performance"
)
def category_performance():
    silver_df = dlt.read("silver_orders")

    gold_agg2 = silver_df.groupBy("category").agg(
        sum("sales_amount").alias("total_sales"),
        count("sale_id").alias("total_orders"),
    )

    return gold_agg2


@dlt.table(
    comment="Gold delta table for store performance"
)
def store_performance():
    silver_df = dlt.read("silver_orders")

    gold_agg3 = silver_df.groupBy("store_number").agg(
        sum("sales_amount").alias("total_sales"),
        count("sale_id").alias("total_orders"),
    )

    return gold_agg3
